## MLflow's Model Registry

In [12]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

### Interacting with the MLflow tracking server

The `MlflowClient` object allows us to interact with...
- an MLflow Tracking Server that creates and manages experiments and runs.
- an MLflow Registry Server that creates and manages registered models and model versions. 

To instantiate it we need to pass a tracking URI and/or a registry URI

In [13]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='/Users/dubai/MLOps_course/mlops_zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1786957730460, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786957730460, lifecycle_stage='active', name='nyx-taxi-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/Users/dubai/MLOps_course/mlops_zoomcamp/02-experiment-tracking/mlruns/0', creation_time=1786957730457, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786957730457, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [ ]:
# client.create_experiment(name="my-cool-experiment")

Let's check the latest versions for the experiment with id `1`...

In [16]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 12",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [17]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 0e8ef8a309a2445687a3ae1f7b06a522, rmse: 8.4820
run id: f5a622aab01142679d17c61bf1916c7e, rmse: 8.4820
run id: f1483938c3b54bceb0aa8ce04e2a262e, rmse: 8.7113
run id: 045ba41ddb804ad7bea0e02cc6eba53f, rmse: 8.7306
run id: cae7e94b8fdb427c9df232b9e641be72, rmse: 9.1275


### Interacting with the Model Registry

In this section We will use the `MlflowClient` instance to:

1. Register a new version for the experiment `nyc-taxi-regressor`
2. Retrieve the latests versions of the model `nyc-taxi-regressor` and check that a new version `4` was created.
3. Transition the version `4` to "Staging" and adding annotations to it.

In [18]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [20]:
run_id = "0e8ef8a309a2445687a3ae1f7b06a522"
model_uri = f"runs:/{run_id}/models_mlflow"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/08/20 12:13:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 0e8ef8a309a2445687a3ae1f7b06a522 has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-6d2fc2f0f5c448cfa2b502b5ceaa53ea instead
Created version '1' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1787213583731, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1787213583731, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='0e8ef8a309a2445687a3ae1f7b06a522', run_link=None, source='models:/m-6d2fc2f0f5c448cfa2b502b5ceaa53ea', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [21]:
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: None


/var/folders/yp/kkdytntx39q1m03lrfdnh19r0000gn/T/ipykernel_9701/669935608.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [23]:
model_version = 1
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/var/folders/yp/kkdytntx39q1m03lrfdnh19r0000gn/T/ipykernel_9701/1600074043.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1787213583731, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1787213600148, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='0e8ef8a309a2445687a3ae1f7b06a522', run_link=None, source='models:/m-6d2fc2f0f5c448cfa2b502b5ceaa53ea', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [24]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1787213583731, current_stage='Staging', deployment_job_state=None, description='The model version 1 was transitioned to Staging on 2026-08-20', last_updated_timestamp=1787213602543, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='0e8ef8a309a2445687a3ae1f7b06a522', run_link=None, source='models:/m-6d2fc2f0f5c448cfa2b502b5ceaa53ea', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

### Comparing versions and selecting the new "Production" model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [28]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime 
    df['duration'] = df['duration'].apply(lambda td: td.total_seconds() / 60)

    df = df[((df.duration >=1) & (df.duration <=60))]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [29]:
df = read_dataframe("/Users/dubai/MLOps_course/mlops_zoomcamp/02-experiment-tracking/data/green_tripdata_2021-03.parquet")

In [30]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/Users/dubai/MLOps_course/mlops_zoomcamp/02-experiment-tracking/preprocessor'

In [31]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [32]:
X_test = preprocess(df, dv)

In [33]:
target = "duration"
y_test = df[target].values

In [35]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

CPU times: user 10.1 s, sys: 109 ms, total: 10.2 s
Wall time: 1.79 s


{'rmse': 8.477060281069058}

In [42]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

CPU times: user 6.94 s, sys: 216 ms, total: 7.16 s
Wall time: 7.28 s


{'rmse': 6.881555517147188}

In [36]:
client.transition_model_version_stage(
    name=model_name,
    version=1,
    stage="Production",
    archive_existing_versions=True
)

/var/folders/yp/kkdytntx39q1m03lrfdnh19r0000gn/T/ipykernel_9701/1316468422.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1787213583731, current_stage='Production', deployment_job_state=None, description='The model version 1 was transitioned to Staging on 2026-08-20', last_updated_timestamp=1787213807780, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='0e8ef8a309a2445687a3ae1f7b06a522', run_link=None, source='models:/m-6d2fc2f0f5c448cfa2b502b5ceaa53ea', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>